# AI Power Portfolio — Colab

Pon la URL de tu repo abajo y corre todas las celdas en orden (Runtime > Run all).

In [ ]:
REPO_URL = "https://github.com/r0988137/Capstone.git"


In [ ]:
import subprocess, os

repo_name = REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
subprocess.run(f"rm -rf {repo_name}", shell=True)
subprocess.run(f"git clone {REPO_URL}", shell=True)

# encuentra la carpeta que tenga src/ y requirements.txt, sin importar el nivel
project_root = None
for dirpath, dirnames, filenames in os.walk(repo_name):
    if "src" in dirnames and "requirements.txt" in filenames:
        project_root = dirpath
        break

assert project_root is not None, "No se encontró src/ + requirements.txt en el repo clonado"
os.chdir(project_root)
print("Proyecto en:", os.getcwd())

!pip install -q -r requirements.txt


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = "/content/drive/MyDrive/ai_power_portfolio_data"
os.makedirs(DRIVE_DIR, exist_ok=True)
os.environ["WAREHOUSE_DB_PATH"] = f"{DRIVE_DIR}/warehouse.db"


In [ ]:
!python -m src.db.init_db
!python -m src.etl.load_market_data


In [ ]:
import sqlite3, pandas as pd

conn = sqlite3.connect(os.environ["WAREHOUSE_DB_PATH"])
df = pd.read_sql("""
    SELECT a.ticker, d.full_date, f.close
    FROM fact_market_price f
    JOIN dim_asset a ON a.asset_key = f.asset_key
    JOIN dim_date d ON d.date_key = f.date_key
    ORDER BY d.full_date DESC
    LIMIT 20
""", conn)
conn.close()
df
